# Imports


In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
from datetime import date as date
import yfinance as yf
import re
from hmmlearn.hmm import GaussianHMM
from pandas_datareader.data import DataReader
import matplotlib.pyplot as plt

# Data Management

In [2]:
# Data Extraction
start_date = '2019-09-18'
end_date = date.today()
symbol = 'HBAR-USD'
data = yf.download(symbol, start_date, end_date)
data

YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,HBAR-USD,HBAR-USD,HBAR-USD,HBAR-USD,HBAR-USD
Date,,,,,
2019-09-18,0.086921,0.100272,0.080045,0.090519,14772274
2019-09-19,0.057924,0.087721,0.054469,0.087702,27324784
2019-09-20,0.052490,0.059061,0.047800,0.058087,15814443
2019-09-21,0.048021,0.055408,0.045456,0.052402,13144171
2019-09-22,0.039735,0.048237,0.038757,0.048065,10549578
...,...,...,...,...,...
2025-03-13,0.189250,0.202045,0.184279,0.200960,418932757
2025-03-14,0.191889,0.198503,0.187881,0.189250,309365663


In [3]:
print(data)

Price          Close      High       Low      Open     Volume
Ticker      HBAR-USD  HBAR-USD  HBAR-USD  HBAR-USD   HBAR-USD
Date                                                         
2019-09-18  0.086921  0.100272  0.080045  0.090519   14772274
2019-09-19  0.057924  0.087721  0.054469  0.087702   27324784
2019-09-20  0.052490  0.059061  0.047800  0.058087   15814443
2019-09-21  0.048021  0.055408  0.045456  0.052402   13144171
2019-09-22  0.039735  0.048237  0.038757  0.048065   10549578
...              ...       ...       ...       ...        ...
2025-03-13  0.189250  0.202045  0.184279  0.200960  418932757
2025-03-14  0.191889  0.198503  0.187881  0.189250  309365663
2025-03-15  0.192706  0.197011  0.190460  0.191885  223717313
2025-03-16  0.185202  0.194836  0.182333  0.192705  271502752
2025-03-17  0.191654  0.195641  0.184834  0.185198  289292129

[2008 rows x 5 columns]


In [5]:
data.head(10)

Price,Close,High,Low,Open,Volume
Ticker,HBAR-USD,HBAR-USD,HBAR-USD,HBAR-USD,HBAR-USD
Date,,,,,
2019-09-18,0.086921,0.100272,0.080045,0.090519,14772274
2019-09-19,0.057924,0.087721,0.054469,0.087702,27324784
2019-09-20,0.052490,0.059061,0.047800,0.058087,15814443
2019-09-21,0.048021,0.055408,0.045456,0.052402,13144171
2019-09-22,0.039735,0.048237,0.038757,0.048065,10549578
2019-09-23,0.037645,0.039803,0.035488,0.039739,11214860
2019-09-24,0.029641,0.043997,0.029084,0.037689,12197101
2019-09-25,0.030202,0.031111,0.025871,0.029515,8906428


In [6]:
data.tail(10)

Price,Close,High,Low,Open,Volume
Ticker,HBAR-USD,HBAR-USD,HBAR-USD,HBAR-USD,HBAR-USD
Date,,,,,
2025-03-08,0.226643,0.235342,0.222927,0.232624,268708260
2025-03-09,0.199269,0.230828,0.198443,0.226653,362384007
2025-03-10,0.189435,0.217289,0.188491,0.199239,486936638
2025-03-11,0.195807,0.203500,0.180726,0.189429,633052141
2025-03-12,0.200960,0.214328,0.193305,0.195807,605417094
2025-03-13,0.189250,0.202045,0.184279,0.200960,418932757
2025-03-14,0.191889,0.198503,0.187881,0.189250,309365663
2025-03-15,0.192706,0.197011,0.190460,0.191885,223717313


In [7]:
data.describe()

Price,Close,High,Low,Open,Volume
Ticker,HBAR-USD,HBAR-USD,HBAR-USD,HBAR-USD,HBAR-USD
count,2008.000000,2008.000000,2008.000000,2008.000000,2.008000e+03
mean,0.115571,0.121145,0.109949,0.115566,1.433483e+08
std,0.103249,0.109544,0.097283,0.103311,3.487073e+08
min,0.010080,0.010667,0.010012,0.010054,4.019320e+05
25%,0.046689,0.048221,0.045390,0.046691,2.052982e+07
50%,0.064723,0.066918,0.062363,0.064709,4.698076e+07
75%,0.178509,0.186428,0.168462,0.178573,1.189548e+08
max,0.505923,0.570146,0.462120,0.505475,6.950736e+09


In [9]:
data.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2008 entries, 2019-09-18 to 2025-03-17
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   (Close, HBAR-USD)   2008 non-null   float64
 1   (High, HBAR-USD)    2008 non-null   float64
 2   (Low, HBAR-USD)     2008 non-null   float64
 3   (Open, HBAR-USD)    2008 non-null   float64
 4   (Volume, HBAR-USD)  2008 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 94.1 KB


In [12]:
def unify_column_names(name: str) -> any:
    """Converts column names to lowercase, replaces spaces/hyphens with underscores, and handles MultiIndex."""
    if isinstance(name, tuple):  # Handle MultiIndex columns
        return tuple(re.sub(r'[-\s]+', '_', n).lower() if isinstance(n, str) else n for n in name)
    return re.sub(r'[-\s]+', '_', name).lower() if isinstance(name, str) else name


# Apply transformation to column names
data.columns = data.columns.map(unify_column_names)

In [13]:
print(f"Unified columns names: \n{data.columns}")

Unified columns names: 
MultiIndex([( 'close', 'hbar_usd'),
            (  'high', 'hbar_usd'),
            (   'low', 'hbar_usd'),
            (  'open', 'hbar_usd'),
            ('volume', 'hbar_usd')],
           names=['Price', 'Ticker'])


In [14]:
# Add Returns and Range
df = data.copy()
df["returns"] = (df["close"] / df["close"].shift(1)) - 1
df["range"] = (df["high"] / df["low"]) - 1

Price,close,high,low,open,volume
Ticker,hbar_usd,hbar_usd,hbar_usd,hbar_usd,hbar_usd
Date,,,,,
2019-09-18,0.086921,0.100272,0.080045,0.090519,14772274
2019-09-19,0.057924,0.087721,0.054469,0.087702,27324784
2019-09-20,0.052490,0.059061,0.047800,0.058087,15814443
2019-09-21,0.048021,0.055408,0.045456,0.052402,13144171
2019-09-22,0.039735,0.048237,0.038757,0.048065,10549578
2019-09-23,0.037645,0.039803,0.035488,0.039739,11214860
2019-09-24,0.029641,0.043997,0.029084,0.037689,12197101
2019-09-25,0.030202,0.031111,0.025871,0.029515,8906428


In [15]:
df.head(10)

Price,close,high,low,open,volume,returns,range
Ticker,hbar_usd,hbar_usd,hbar_usd,hbar_usd,hbar_usd,,
Date,,,,,,,
2019-09-18,0.086921,0.100272,0.080045,0.090519,14772274,NaN,0.252695
2019-09-19,0.057924,0.087721,0.054469,0.087702,27324784,-0.333602,0.610476
2019-09-20,0.052490,0.059061,0.047800,0.058087,15814443,-0.093813,0.235586
2019-09-21,0.048021,0.055408,0.045456,0.052402,13144171,-0.085140,0.218937
2019-09-22,0.039735,0.048237,0.038757,0.048065,10549578,-0.172549,0.244601
2019-09-23,0.037645,0.039803,0.035488,0.039739,11214860,-0.052598,0.121590
2019-09-24,0.029641,0.043997,0.029084,0.037689,12197101,-0.212618,0.512756
2019-09-25,0.030202,0.031111,0.025871,0.029515,8906428,0.018926,0.202543


In [17]:
# Determine total of NA values
print(f"Total NA Values: {df.isnull().sum().sum()}")

Total NA Values: 1


In [18]:
# Drop NA values
df.dropna(inplace=True)
df.head(10)

Price,close,high,low,open,volume,returns,range
Ticker,hbar_usd,hbar_usd,hbar_usd,hbar_usd,hbar_usd,,
Date,,,,,,,
2019-09-19,0.057924,0.087721,0.054469,0.087702,27324784,-0.333602,0.610476
2019-09-20,0.052490,0.059061,0.047800,0.058087,15814443,-0.093813,0.235586
2019-09-21,0.048021,0.055408,0.045456,0.052402,13144171,-0.085140,0.218937
2019-09-22,0.039735,0.048237,0.038757,0.048065,10549578,-0.172549,0.244601
2019-09-23,0.037645,0.039803,0.035488,0.039739,11214860,-0.052598,0.121590
2019-09-24,0.029641,0.043997,0.029084,0.037689,12197101,-0.212618,0.512756
2019-09-25,0.030202,0.031111,0.025871,0.029515,8906428,0.018926,0.202543
2019-09-26,0.028966,0.031022,0.027305,0.030204,5190095,-0.040924,0.136129
